## TO DO:
- Check configuratie; wat is het beste voor MA detectie?
- Check Morlet vs. Welch of andere?
- Check alle berekeningen -> slow/fast ratio / activation score
    - Wat betekenen deze scores?

- Maak algoritme zo dat je T & N zelf kan invullen
- Zet alle bestanden die je daadwerkelijk gaat gebruiken in een aparte map bij elkaar zodat je niet onnodig dingen gebruikt
    - Of blokkeer in algoritme dat de bestanden die niet meedoen worden gebruikt (ik denk dat dit makkelijker is eig)

In [2]:
import mne
import numpy as np
import pandas as pd
import gc
from pathlib import Path

In [4]:
import pandas as pd

stage_file = r"\\vs03.herseninstituut.knaw.nl\VS03-SandC-2\raw\bnbd\Data\eeg\NSR\bnbd_nsr_03554\bnbd_nsr_03554_T0_N1\sleepArchitecture\bnbd_nsr_03554_T0_N1.csv"

df_stages = pd.read_csv(stage_file)
print(df_stages.head(20))
print()
print(df_stages.columns.tolist())
print(df_stages.shape)

    0  0.1
0   0    0
1   0    0
2   0    0
3   0    0
4   0    0
5   0    0
6   0    0
7   0    0
8   0    0
9   0    0
10  0    0
11  0    0
12  0    0
13  0    0
14  0    0
15  0    0
16  0    0
17  0    0
18  0    0
19  0    0

['0', '0.1']
(961, 2)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Configuratie
# ══════════════════════════════════════════════════════════════════════════════
base_dir   = Path(r"\\vs03.herseninstituut.knaw.nl\VS03-SandC-2\raw\bnbd\Data\eeg\NSR")
output_dir = Path(r"C:\Users\zafar\Documents\bnbd_output2")
output_dir.mkdir(exist_ok=True)

MAX_PARTICIPANTS = 5

EEG_CH = ['EEG L psg-lp', 'EEG R psg-lp']
EMG_CH = ['EEG L psg-emg', 'EEG R psg-emg']
MOV_CH = ['dX', 'dY', 'dZ']
ALL_CH = EEG_CH + EMG_CH + MOV_CH

SFREQ         = 256.0
WIN_SEC       = 1.0
STEP_SEC      = 0.5
WIN_SAMP      = int(WIN_SEC  * SFREQ)
STEP_SAMP     = int(STEP_SEC * SFREQ)
CHUNK_MINUTES = 5
CHUNK_SAMP    = int(CHUNK_MINUTES * 60 * SFREQ)

FREQS = np.arange(0.5, 35.5, 0.5)

BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0,  13.0),
    'beta':  (13.0, 35.0),
}

ROLLING_SEC            = 60.0
AROUSAL_FREQ_THRESHOLD = 8.0
AROUSAL_MIN_DUR        = 3.0
AROUSAL_MAX_DUR        = 30.0


# ══════════════════════════════════════════════════════════════════════════════
# Morlet wavelet (van supervisor)
# ══════════════════════════════════════════════════════════════════════════════
def compute_morlet_tf(signal, srate, freqs, n_cycles=None, L2normalize=False):
    freqs = np.asarray(freqs)
    if n_cycles is None:
        n_cycles_arr = np.maximum(3.0, freqs / 2.0)
    elif np.isscalar(n_cycles):
        n_cycles_arr = np.full(len(freqs), float(n_cycles))
    else:
        n_cycles_arr = np.asarray(n_cycles, dtype=float)

    n_samples  = len(signal)
    signal     = signal - np.mean(signal)
    signal_fft = np.fft.fft(signal)
    fft_freqs  = np.fft.fftfreq(n_samples, d=1.0 / srate)

    power = np.empty((len(freqs), n_samples), dtype=np.float32)
    for i, freq in enumerate(freqs):
        sigma_f     = freq / n_cycles_arr[i]
        wavelet_fft = np.exp(-0.5 * ((fft_freqs - freq) / sigma_f) ** 2)
        if L2normalize:
            wavelet_fft /= np.sqrt(np.sum(wavelet_fft ** 2))
        analytic = np.fft.ifft(signal_fft * wavelet_fft)
        power[i] = np.abs(analytic) ** 2

    return power  # shape: (n_freqs, n_samples)


def band_mean(power, freqs, fmin, fmax):
    mask = (freqs >= fmin) & (freqs <= fmax)
    return power[mask, :].mean(axis=0)


# ══════════════════════════════════════════════════════════════════════════════
# Fase 1 — Load & preprocess
# ══════════════════════════════════════════════════════════════════════════════
def load_night(edf_file):
    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose=False)
    raw.pick(ALL_CH)
    raw.load_data(verbose=False)
    raw._data = raw._data.astype(np.float64)

    raw.filter(l_freq=0.5, h_freq=35.0, picks=EEG_CH, verbose=False)
    h_emg = min(100.0, SFREQ / 2 - 1)
    raw.filter(l_freq=10.0, h_freq=h_emg, picks=EMG_CH, verbose=False)
    raw.apply_function(lambda x: x - np.mean(x), picks=MOV_CH, verbose=False)

    return raw


def preprocess_signals(raw):
    signals = {}
    for ch in ALL_CH:
        signals[ch] = raw.get_data(picks=ch)[0]
    return signals


# ══════════════════════════════════════════════════════════════════════════════
# Fase 2 — Feature extractie via Morlet
# ══════════════════════════════════════════════════════════════════════════════
def extract_features_morlet(signals, n_total):
    starts = np.arange(0, n_total - WIN_SAMP + 1, STEP_SAMP)
    df     = pd.DataFrame({'time_sec': starts / SFREQ})

    # ── EEG ──────────────────────────────────────────────────────────────────
    for ch in EEG_CH:
        tag    = 'L' if 'L' in ch else 'R'
        signal = signals[ch]
        print(f"    Morlet: {ch}")

        band_ts = {b: np.zeros(n_total, dtype=np.float64) for b in BANDS}

        offset = 0
        while offset < n_total:
            end   = min(offset + CHUNK_SAMP, n_total)
            chunk = signal[offset:end].astype(np.float64)
            power = compute_morlet_tf(chunk, srate=SFREQ, freqs=FREQS,
                                      L2normalize=True)
            for band_name, (fmin, fmax) in BANDS.items():
                band_ts[band_name][offset:end] = band_mean(power, FREQS, fmin, fmax)
            del power, chunk
            offset += CHUNK_SAMP

        for band_name in BANDS:
            col_vals = []
            for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
                col_vals.append(float(band_ts[band_name][s:s + WIN_SAMP].mean()))
            df[f'eeg_{tag}_{band_name}'] = col_vals

        rms_vals, ll_vals = [], []
        for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
            seg = signal[s:s + WIN_SAMP]
            rms_vals.append(float(np.sqrt(np.mean(seg ** 2))))
            ll_vals.append(float(np.sum(np.abs(np.diff(seg)))))

        df[f'eeg_{tag}_rms']             = rms_vals
        df[f'eeg_{tag}_line_length']     = ll_vals
        fast = df[f'eeg_{tag}_alpha'] + df[f'eeg_{tag}_beta']
        slow = df[f'eeg_{tag}_delta'] + df[f'eeg_{tag}_theta'] + 1e-12
        df[f'eeg_{tag}_fast_slow_ratio'] = fast / slow

        del band_ts

    # ── EMG ──────────────────────────────────────────────────────────────────
    for ch in EMG_CH:
        tag    = 'L' if 'L' in ch else 'R'
        signal = signals[ch]
        rms_vals = []
        for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
            seg = signal[s:s + WIN_SAMP]
            rms_vals.append(float(np.sqrt(np.mean(seg ** 2))))
        df[f'emg_{tag}_rms'] = rms_vals

    # ── Beweging ──────────────────────────────────────────────────────────────
    for ch in MOV_CH:
        axis   = ch[-1].lower()
        signal = signals[ch]
        rms_vals = []
        for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
            seg = signal[s:s + WIN_SAMP]
            rms_vals.append(float(np.sqrt(np.mean(seg ** 2))))
        df[f'mov_{axis}_rms'] = rms_vals

    return df


# ══════════════════════════════════════════════════════════════════════════════
# Fase 3 — Local baseline normalisatie
# ══════════════════════════════════════════════════════════════════════════════
def normalise_features(df):
    roll_rows = int(ROLLING_SEC / STEP_SEC)  # 120 rijen

    eeg_feature_cols = [
        c for c in df.columns
        if c.startswith('eeg_') and not c.endswith('_z')
    ]

    df_norm = df.copy()

    for col in eeg_feature_cols:
        roll_med = df[col].rolling(roll_rows, min_periods=1, center=False).median()
        roll_mad = (
            df[col]
            .rolling(roll_rows, min_periods=1, center=False)
            .apply(lambda x: np.median(np.abs(x - np.median(x))), raw=True)
        )
        z = (df[col] - roll_med) / (roll_mad.clip(lower=1e-3) + 1e-6)
        df_norm[f'{col}_z'] = z.clip(-10, 10)

    z_cols = [f'{c}_z' for c in eeg_feature_cols]
    df_norm['activation_score'] = df_norm[z_cols].mean(axis=1)

    return df_norm

# ══════════════════════════════════════════════════════════════════════════════
# Fase 4 — Bijvoegen slaapstadia als beschikbaar
# ══════════════════════════════════════════════════════════════════════════════

def load_sleep_stages(pid, participant_folder):
    """Laad slaapstadia CSV en geef een Series terug met epoch_nr als index."""
    stage_file = (
        participant_folder
        / f"bnbd_nsr_{pid}_T0_N1"
        / "sleepArchitecture"
        / f"bnbd_nsr_{pid}_T0_N1.csv"
    )

    if not stage_file.exists():
        print(f"  [{pid}] Geen slaapstadia bestand gevonden.")
        return None

    df_stages = pd.read_csv(stage_file, header=None, usecols=[0])
    df_stages.columns = ['stage_raw']

    # Vertaal naar leesbare labels
    stage_map = {0: 'W', 1: 'N1', 2: 'N2', 3: 'N3', 5: 'REM'}
    df_stages['stage'] = df_stages['stage_raw'].map(stage_map)
    df_stages['epoch_nr'] = df_stages.index  # rij 0 = epoch 0 = seconde 0-30

    return df_stages[['epoch_nr', 'stage']]


def add_sleep_stages(df_features, df_stages):
    """Voeg slaapstadium toe aan features dataframe via epoch nummer."""
    df_features['epoch_nr'] = (df_features['time_sec'] // 30).astype(int)

    if df_stages is not None:
        df_features = df_features.merge(df_stages, on='epoch_nr', how='left')
    else:
        df_features['stage'] = 'unknown'

    return df_features


# ══════════════════════════════════════════════════════════════════════════════
# Hoofdloop
# ══════════════════════════════════════════════════════════════════════════════
for participant_folder in sorted(base_dir.glob("bnbd_nsr_?????"))[:MAX_PARTICIPANTS]:
    pid = participant_folder.name.split("_")[-1]

    edf_file = (
        participant_folder
        / f"bnbd_nsr_{pid}_T0_N1"
        / "sleepArchitecture"
        / f"bnbd_nsr_{pid}_T0_N1_psg.edf"
    )

    if not edf_file.exists():
        print(f"[{pid}] Niet gevonden, overgeslagen.")
        continue

    out_path = output_dir / f"features_{pid}.csv"
    if out_path.exists():
        print(f"[{pid}] Al verwerkt, overgeslagen.")
        continue

    try:
        print(f"\n[{pid}] ── Fase 1: laden & preprocessen...")
        raw     = load_night(edf_file)
        signals = preprocess_signals(raw)
        n_total = raw.n_times
        del raw
        gc.collect()

        print(f"[{pid}] ── Fase 2: feature extractie (Morlet)...")
        df_feat = extract_features_morlet(signals, n_total)
        del signals
        gc.collect()

        print(f"[{pid}] ── Fase 3: normalisatie...")
        df_norm = normalise_features(df_feat)
        del df_feat
        gc.collect()

        print(f"[{pid}] ── Fase 4: slaapstadia koppelen...")
        df_stages = load_sleep_stages(pid, participant_folder)
        df_norm   = add_sleep_stages(df_norm, df_stages)

        df_norm.to_csv(out_path, index=False, sep=',', decimal='.', float_format='%.6f')
        print(f"[{pid}] Opgeslagen: {out_path.name}  ({len(df_norm)} vensters)")
        del df_norm
        gc.collect()

    except MemoryError:
        print(f"[{pid}] MemoryError - overgeslagen.")
        gc.collect()
        continue

print("\nFase 1–3 klaar.")

[00881] Niet gevonden, overgeslagen.
[01272] Al verwerkt, overgeslagen.
[01614] Niet gevonden, overgeslagen.
[03554] Al verwerkt, overgeslagen.
[03983] Al verwerkt, overgeslagen.

Fase 1–3 klaar.


In [ ]:
from pathlib import Path
import os

pad = Path("C:\\Users\\zafar\\Documents\\bnbd_output2")
os.chdir(pad)

pd.set_option('display.max_columns', None)
df = pd.read_csv('features_03554.csv', sep=',', decimal=',')
df.head(10)


,time_sec,eeg_L_delta,eeg_L_theta,eeg_L_alpha,eeg_L_beta,eeg_L_rms,eeg_L_line_length,eeg_L_fast_slow_ratio,eeg_R_delta,eeg_R_theta,eeg_R_alpha,eeg_R_beta,eeg_R_rms,eeg_R_line_length,eeg_R_fast_slow_ratio,emg_L_rms,emg_R_rms,mov_x_rms,mov_y_rms,mov_z_rms,eeg_L_delta_z,eeg_L_theta_z,eeg_L_alpha_z,eeg_L_beta_z,eeg_L_rms_z,eeg_L_line_length_z,eeg_L_fast_slow_ratio_z,eeg_R_delta_z,eeg_R_theta_z,eeg_R_alpha_z,eeg_R_beta_z,eeg_R_rms_z,eeg_R_line_length_z,eeg_R_fast_slow_ratio_z,activation_score,epoch_nr,stage
0,0.000000,0.278265,0.012143,0.002597,0.000476,11.339407,336.866210,0.010580,0.093345,0.012433,0.003926,0.001343,12.887468,553.700045,0.049815,4.178763,7.173397,0.037054,0.046141,0.097722,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,W
1,0.500000,0.719861,0.013955,0.001640,0.000373,35.533618,331.734122,0.002744,0.190826,0.009969,0.002863,0.001357,23.449687,556.665690,0.021014,3.552424,6.812495,0.047876,0.060880,0.172316,0.999995,0.904905,-0.477602,-0.051300,1.000000,-1.000000,-0.999745,0.999979,-0.999189,-0.530895,0.006531,1.000000,0.999999,-0.999931,0.060911,0,W
2,1.000000,1.301730,0.033379,0.002474,0.001327,63.785809,492.743068,0.002846,0.249685,0.027696,0.002244,0.001407,29.318009,610.207731,0.013164,6.726339,6.893176,0.088321,0.089740,0.251123,1.317648,10.000000,0.000000,0.850085,1.167725,10.000000,0.000000,0.999983,6.192986,-0.617950,0.050553,1.000000,10.000000,-0.999873,2.854368,0,W
3,1.500000,1.512729,0.068566,0.013169,0.002841,65.108270,620.563155,0.010125,0.185263,0.030237,0.003757,0.001541,22.806531,570.118341,0.024582,9.202096,6.994024,0.094108,0.084615,0.287899,1.266118,4.228209,10.000000,1.937960,1.044716,2.555864,0.985870,-0.086358,1.147582,0.446324,0.158980,-0.098773,0.819369,0.312471,1.765595,0,W
4,2.000000,1.000785,0.043742,0.011393,0.001880,39.292683,412.703469,0.012707,0.079893,0.008834,0.002662,0.000965,12.703990,338.825486,0.040883,7.192590,5.315653,0.061317,0.056372,0.256019,0.000000,0.533474,8.787320,0.552905,0.000000,0.000000,0.999613,-1.635588,-0.999722,-0.200740,-0.390772,-1.551497,-10.000000,1.427508,-0.176964,0,W
5,2.500000,0.413716,0.002419,0.000487,0.000195,17.906704,199.045858,0.001638,0.037851,0.002511,0.000722,0.000152,11.194011,158.840215,0.021664,2.691609,2.019942,0.046347,0.043273,0.193006,-1.005854,-1.344781,-1.391122,-0.705510,-0.855918,-2.182972,-1.236854,-1.829038,-1.571469,-2.037961,-1.196904,-1.238261,-10.000000,-0.241804,-1.917032,0,W
6,3.000000,0.193463,0.004431,0.001806,0.000239,15.790248,221.521317,0.010331,0.035788,0.003544,0.000712,0.000141,12.150189,171.025570,0.021695,2.797128,2.006864,0.032282,0.015044,0.135536,-1.192034,-0.825448,-0.667199,-0.236943,-1.000000,-1.000000,0.079692,-0.999983,-0.999844,-1.779705,-1.201366,-0.435369,-6.772078,0.000000,-1.216448,0,W
7,3.500000,0.117080,0.007956,0.002456,0.000347,15.553509,280.873078,0.022412,0.069046,0.009002,0.001945,0.000170,14.623505,206.394983,0.027100,3.485208,2.652342,0.020276,0.011044,0.084290,-1.114071,-0.529170,-0.008872,-0.077952,-0.940778,-0.558913,2.470718,-0.352866,-0.108850,-0.507683,-0.983350,0.416638,-1.666903,1.301432,-0.190044,0,W
8,4.000000,0.126364,0.016455,0.003417,0.000479,18.839953,324.110458,0.027279,0.141688,0.015056,0.003570,0.000279,19.931717,252.460793,0.024556,4.060833,3.481086,0.014859,0.041722,0.049595,-0.968702,0.262559,0.942911,0.002626,0.000000,-0.094155,2.264207,0.871134,0.999803,0.907124,-0.685749,1.547812,-0.479843,0.000000,0.397838,0,W
9,4.500000,0.146318,0.023432,0.002937,0.000381,17.955870,289.516349,0.019542,0.247298,0.033490,0.003238,0.000276,21.818289,284.452163,0.012514,3.558953,3.425937,0.018720,0.047034,0.068295,-0.890326,0.865899,0.401158,-0.047600,-0.089275,-0.582657,1.186077,1.840112,3.871713,0.474599,-0.345704,0.936122,-0.185314,-3.485902,0.282064,0,W
